# Argument dependent lookup

In [ ]:
#include <iostream>

Consider a simple class called `AClass` defined inside a namespace called `Surrounding`, and some other functions defined inside the same namespace...

In [ ]:
namespace Surrounding {
    struct AClass {};
    void one_func(int x, int y) {
        std::cout << "Calling Surrounding::one_func(int, int) with arguments " << x << " and " << y << "\n";
    }
    void another_func(int x, AClass y) {
        std::cout << "Calling Surrounding::another_func(int, AClass) with arguments " << x << " and an instance of the empty class AClass\n";
    }
}


To create objects of the type `Surrounding::AClass` one has to use the usual namespace rules...

In [ ]:
Surrounding::AClass obj;

Similarly, if you want to call a function defined inside the namespace, you would use the fully qualified name or a `using namespace` declaration...

In [ ]:
Surrounding::one_func(1, 3);

Trying to call that function without using its full name leads to an obvious error...

In [ ]:
one_func(1,3);

While we are at it, why not try that with the other function we wrote inside that namespace using the object called `obj` we created above!

In [ ]:
another_func(1, obj);

Do you observe the difference in behaviour? This is not an error, but an important feature of C++ classes. It's called ADL, or Argument Dependent Lookup.

One would think that, we have an insufficiently specified function called `another_func` above. But one of the arguments we are passing is of the user defined type `Surrounding::AClass`. ADL states that the search for a matching function includes functions defined in the namespaces surrounding the declaration of the user defined type. `another_func` is found only because one of the arguments is of a class type, of a class that was defined inside a namespace. 

This courtesy of looking for a matching function does not extend beyond the namespace(s) surrounding the classes of the parameters.

In [ ]:
namespace NotSurrounding {
    void a_third_func(int x, Surrounding::AClass y)
    {
        std::cout << "Calling NotSurrounding::a_third_func(int, AClass) with arguments " << x << " and an instance of the empty class AClass\n";
    }
}

In [ ]:
a_third_func(1., obj);

Even functions defined inside the same namespace hierarchy, but higher up than the immediate surrounding namespace are not considered...

To demostrate this, we will define a new function called `another_func`, but in the `GrandParent` namespace. Remember, `Surrounding::another_func` defined earlier in this Notebook still exists, and can be used...

In [ ]:
Surrounding::another_func(3, obj);

In [ ]:
namespace GrandParent {
    namespace Surrounding {
        struct AClass {};
    }
    void another_func(int x, Surrounding::AClass y) {
        std::cout << "Calling GrandParent::another_func(int, AClass) with arguments " << x << " and an instance of the empty class AClass\n";
    }
    void another_func1(int x, Surrounding::AClass y) {
        std::cout << "Calling GrandParent::another_func(int, AClass) with arguments " << x << " and an instance of the empty class AClass\n";
    }
}

In [ ]:
another_func(3., obj);

Above: it is as if the function `GrandParent::another_func` didn't exist! If that were visible, we would have an error because of ambiguity.

In [ ]:
another_func1(3., obj);

In case there are other parameters passed to the function which are also of some user defined type, like `NotSurrounding::BClass` below, the namespace around those classes are also searched for a matching function.

In [ ]:
namespace NotSurrounding {
    class BClass {};
    void another_func(Surrounding::AClass x, BClass y) {
        std::cout << "Calling NotSurrounding::another_func(AClass, BClass)\n";
    }
}


In [ ]:
NotSurrounding::BClass obj2;

In [ ]:
another_func(obj, obj2);

ADL looks at the arguments passed to a function and tries to find a matching function defined in any of the namespaces surrounding the arguments. In each case, the immediate surrounding namespace is considered. So, you see, the namespace surrounding a class have a special status!

## Recommendation:
Use ADL! Write a minimal number of functions in each class as member functions. Those should be the functions which can not be implemented (efficiently) without direct access to the member variables. The remaining functions should be in the immediately srrounding namespace, implemented using trivial accessor member functions when necessary. ADL will make those functions available whenever we are using objects of that type!